In [1]:
!pip install -q pytesseract pdf2image pillow opencv-python anthropic
!apt-get install -q poppler-utils
!apt-get install -q tesseract-ocr-ara

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 11.4 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 1s (241 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...
Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be ins

In [2]:
from footer_detector_v3 import FooterDetector, FooterType
from enhanced_stitching_v3 import process_boundary_enhanced

In [3]:
#claude_model = "claude-sonnet-4-20250514"
claude_model = "claude-haiku-4-5-20251001"

In [4]:
import os

output_dir = 'ara-ocr'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
print(f"Created directory: {output_dir}")

Created directory: ara-ocr


In [5]:
import cv2
import numpy as np
import pytesseract
import re
from pdf2image import convert_from_path
from PIL import Image

def optimize_pdf_ocr(pdf_path, lang='ara', dpi=300):
    pages = convert_from_path(pdf_path, dpi)
    full_text = []
    for i, page in enumerate(pages):
        open_cv_image = np.array(page)
        image_rgb = cv2.cvtColor(open_cv_image, cv2.COLOR_RGB2BGR)
        image_rgb = cv2.cvtColor(image_rgb, cv2.COLOR_BGR2RGB)
        text = pytesseract.image_to_string(image_rgb, lang=lang)
        full_text.append(f"--- Page {i+1} ---\n{text}")
    return "\n".join(full_text)

pdf_file = 'pages_5_9.pdf'
optimized_result = optimize_pdf_ocr(pdf_file)

with open(os.path.join(output_dir, 'optimized_approach.txt'), 'w', encoding='utf-8') as f:
    f.write(optimized_result)
print('OCR complete.')

OCR complete.


In [6]:
import anthropic
from google.colab import userdata

ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

def refine_with_llm(raw_text, model_name=claude_model):
    prompt = f"""
أنت خبير في تحقيق النصوص التاريخية. المهمة هي معالجة نص OCR من مذكرات جعفر العسكري.

المطلوب تنفيذ القواعد التالية بدقة:
1. دمج الأسطر (Flowing Text): قم بإلغاء الفواصل المصطنعة بين الأسطر الناتجة عن التنسيق العمودي للكتاب، واجعل النص يتدفق كفقرات متصلة وطبيعية.
2. إصلاح الأخطاء فقط: صحح الكلمات المشوهة آليّاً دون تغيير الأسلوب أو إضافة كلمات.
3. حافظ على الأسماء التاريخية والجغرافية كما وردت.
4. لا تضع نقطة نهاية إذا انتهى النص بجملة مفتوحة.
5. العناوين والأسماء الرئيسية تبقى في أسطر مستقلة.
6. أخرِج النص المصحح مباشرةً فقط، بدون أي عنوان أو تعليق في البداية.

النص الخام:
{raw_text}
    """
    try:
        message = client.messages.create(
            model=model_name,
            max_tokens=2048,
            messages=[{"role": "user", "content": prompt}]
        )
        return message.content[0].text
    except Exception as e:
        print(f"  Warning: {e}")
        return raw_text

page_blocks = re.split(r'(?=--- Page \d+ ---)', optimized_result)
corrected_pages = []
for block in page_blocks:
    block = block.strip()
    if not block:
        continue
    m = re.match(r'(--- Page \d+ ---)\n?(.*)', block, re.DOTALL)
    if m:
        header, page_text = m.group(1), m.group(2).strip()
        if page_text:
            print(f"Correcting {header}...")
            corrected = refine_with_llm(page_text)
            corrected_pages.append(f"{header}\n{corrected}")
        else:
            corrected_pages.append(block)
    else:
        corrected_pages.append(block)

clean_text = "\n\n".join(corrected_pages)
with open(os.path.join(output_dir, 'LLM_corrected_text.txt'), 'w', encoding='utf-8') as f:
    f.write(clean_text)
print('Per-page correction done. Saved LLM_corrected_text.txt')

Correcting --- Page 1 ---...
Correcting --- Page 2 ---...
Correcting --- Page 3 ---...
Correcting --- Page 4 ---...
Correcting --- Page 5 ---...
Per-page correction done. Saved LLM_corrected_text.txt


In [7]:
# =============================================================
# Cell 05
# FOOTER DETECTION & STRIPPING MODULE
# =============================================================
# Paste this cell after the LLM correction cell (Cell 5)

import re
from dataclasses import dataclass
from typing import List, Optional, Tuple
from enum import Enum


class FooterType(Enum):
    PAGE_NUMBER = "page_number"
    FOOTNOTE = "footnote"
    RUNNING_HEADER = "running_header"
    FOOTER_TEXT = "footer_text"
    SEPARATOR = "separator"
    UNKNOWN = "unknown"


@dataclass
class DetectedFooter:
    text: str
    footer_type: FooterType
    confidence: float
    page_num: int
    line_index: int
    original_line: str
    is_stripped: bool = False


class FooterDetector:
    """Detects and classifies footer elements in Arabic OCR text."""

    def __init__(self, page_height_ratio: float = 0.15, min_footer_lines: int = 1):
        self.page_height_ratio = page_height_ratio
        self.min_footer_lines = min_footer_lines
        self.detected_footers: List[DetectedFooter] = []

    def _is_page_number(self, text: str) -> Tuple[bool, float]:
        stripped = text.strip()
        if not stripped:
            return False, 0.0
        # Arabic-Indic digits
        if re.fullmatch(r'[\u0660-\u0669]+', stripped):
            return True, 0.95
        # Eastern Arabic-Indic digits
        if re.fullmatch(r'[\u06F0-\u06F9]+', stripped):
            return True, 0.95
        # Western digits
        if re.fullmatch(r'[0-9]+', stripped):
            return True, 0.90
        # Decorative: - ٥ -
        if re.fullmatch(r'[-\u2013\u2014\s]*[\u0660-\u0669\u06F0-\u06F90-9]+[-\u2013\u2014\s]*', stripped):
            return True, 0.85
        # Arabic word for page + number
        if re.search(r'\u0635\u0641\u062D\u0629?\s*[\u0660-\u0669\u06F0-\u06F90-9]+', stripped):
            return True, 0.90
        return False, 0.0

    def _is_footnote(self, text: str) -> Tuple[bool, float]:
        stripped = text.strip()
        if not stripped:
            return False, 0.0
        # Parenthesized number: (١)
        if re.match(r'^[\(\[\{]\s*[\u0660-\u0669\u06F0-\u06F90-9]\s*[\)\]\}]', stripped):
            return True, 0.95
        # Asterisk, dagger, etc.
        if re.match(r'^[*†‡§¶#\+\-—]', stripped):
            return True, 0.85
        # Arabic letter + parenthesis
        if re.match(r'^[\u0621-\u064A]\)', stripped):
            return True, 0.70
        # Reference markers
        if len(stripped) < 50 and any(m in stripped for m in ['\u0627\u0646\u0638\u0631', '\u0631\u0627\u062C\u0639', '\u0647\u0627\u0645\u0634']):
            return True, 0.60
        return False, 0.0

    def _is_separator(self, text: str) -> Tuple[bool, float]:
        stripped = text.strip()
        if not stripped:
            return False, 0.0
        if re.fullmatch(r'[-_*=\u2014\u2013]+', stripped):
            return True, 0.90
        if re.fullmatch(r'[-_*=\u2014\u2013\s]*[\u0660-\u0669\u06F0-\u06F90-9]+[-_*=\u2014\u2013\s]*', stripped):
            return True, 0.75
        return False, 0.0

    def _is_running_header(self, text: str) -> Tuple[bool, float]:
        stripped = text.strip()
        if not stripped:
            return False, 0.0
        if len(stripped) < 60 and not any(c in stripped for c in '.\u060C:\u061B'):
            if re.search(r'\u0645\u0642\u062F\u0645\u0629|\u0641\u0635\u0644|\u0643\u062A\u0627\u0628|\u0630\u0643\u0631\u064A\u0627\u062A|\u0645\u0630\u0643\u0631\u0627\u062A', stripped):
                return True, 0.80
            return True, 0.50
        return False, 0.0

    def analyze_page(self, page_text: str, page_num: int) -> List[DetectedFooter]:
        lines = page_text.split('\n')
        footers = []
        total_lines = len(lines)
        footer_start_idx = int(total_lines * (1 - self.page_height_ratio))

        for idx in range(footer_start_idx, total_lines):
            line = lines[idx]
            if not line.strip():
                continue
            is_page_num, conf_pn = self._is_page_number(line)
            if is_page_num:
                footers.append(DetectedFooter(text=line.strip(), footer_type=FooterType.PAGE_NUMBER,
                    confidence=conf_pn, page_num=page_num, line_index=idx, original_line=line))
                continue
            is_footnote, conf_fn = self._is_footnote(line)
            if is_footnote:
                footers.append(DetectedFooter(text=line.strip(), footer_type=FooterType.FOOTNOTE,
                    confidence=conf_fn, page_num=page_num, line_index=idx, original_line=line))
                continue
            is_sep, conf_sep = self._is_separator(line)
            if is_sep:
                footers.append(DetectedFooter(text=line.strip(), footer_type=FooterType.SEPARATOR,
                    confidence=conf_sep, page_num=page_num, line_index=idx, original_line=line))
                continue

        for idx in range(int(total_lines * 0.15)):
            line = lines[idx]
            if not line.strip():
                continue
            is_rh, conf_rh = self._is_running_header(line)
            if is_rh:
                footers.append(DetectedFooter(text=line.strip(), footer_type=FooterType.RUNNING_HEADER,
                    confidence=conf_rh, page_num=page_num, line_index=idx, original_line=line))

        footers = self._link_footnote_continuations(lines, footers, page_num)
        return footers

    def _link_footnote_continuations(self, lines, footers, page_num):
        fn_markers = [f for f in footers if f.footer_type == FooterType.FOOTNOTE]
        if not fn_markers:
            return footers
        for fn in fn_markers:
            next_idx = fn.line_index + 1
            while next_idx < len(lines) and next_idx < fn.line_index + 5:
                next_line = lines[next_idx].strip()
                if not next_line:
                    break
                if not re.match(r'^[\(\[\{]\s*[\u0660-\u0669\u06F0-\u06F90-9]\s*[\)\]\}]', next_line):
                    if len(next_line) < 120 or not next_line.endswith('.'):
                        footers.append(DetectedFooter(text=next_line, footer_type=FooterType.FOOTNOTE,
                            confidence=0.60, page_num=page_num, line_index=next_idx, original_line=lines[next_idx]))
                        next_idx += 1
                    else:
                        break
                else:
                    break
        return footers

    def strip_footers(self, page_text: str, footers: List[DetectedFooter],
                     preserve_types: Optional[List[FooterType]] = None) -> str:
        if preserve_types is None:
            preserve_types = []
        lines = page_text.split('\n')
        indices_to_remove = set()
        for footer in footers:
            if footer.footer_type not in preserve_types:
                indices_to_remove.add(footer.line_index)
                footer.is_stripped = True
        cleaned_lines = [line for idx, line in enumerate(lines) if idx not in indices_to_remove]
        return '\n'.join(cleaned_lines)

    def get_footer_report(self) -> str:
        if not self.detected_footers:
            return "No footers detected."
        report = ["=== FOOTER DETECTION REPORT ===", ""]
        current_page = 0
        for footer in self.detected_footers:
            if footer.page_num != current_page:
                current_page = footer.page_num
                report.append(f"\n--- Page {current_page} ---")
            status = "STRIPPED" if footer.is_stripped else "PRESERVED"
            report.append(f"  [{footer.footer_type.value}] (conf: {footer.confidence:.2f}) {status}: {footer.text[:60]}")
        return '\n'.join(report)

print("Footer detection module loaded.")


Footer detection module loaded.


In [8]:
# =============================================================
# Cell 06
# IMAGE-BASED FOOTER DETECTION (Optional)
# =============================================================
# Uses OpenCV for visual footer detection before OCR.
# Slower but more accurate for complex layouts.

import cv2
import numpy as np
from pdf2image import convert_from_path
from PIL import Image
from typing import Tuple, Dict

class ImageFooterDetector:
    def __init__(self, dpi: int = 300, footer_ratio: float = 0.12):
        self.dpi = dpi
        self.footer_ratio = footer_ratio

    def detect_footer_region(self, pil_image: Image.Image) -> Tuple[int, int, int, int]:
        img_array = np.array(pil_image)
        height, width = img_array.shape[:2]
        if len(img_array.shape) == 3:
            gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
        else:
            gray = img_array
        bottom_start = int(height * (1 - self.footer_ratio * 2))
        bottom_region = gray[bottom_start:, :]
        thresh = cv2.adaptiveThreshold(bottom_region, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY_INV, 11, 2)
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        footer_contours = []
        for cnt in contours:
            x, y, w, h = cv2.boundingRect(cnt)
            area = w * h
            if 50 < area < width * height * 0.05:
                footer_contours.append((x, y + bottom_start, w, h))
        if not footer_contours:
            return (0, height - int(height * self.footer_ratio), width, int(height * self.footer_ratio))
        min_x = min(c[0] for c in footer_contours)
        min_y = min(c[1] for c in footer_contours)
        max_x = max(c[0] + c[2] for c in footer_contours)
        max_y = max(c[1] + c[3] for c in footer_contours)
        return (min_x, min_y, max_x - min_x, max_y - min_y)

print("Image-based footer detector loaded.")

Image-based footer detector loaded.


In [9]:
# =============================================================
# Cell 07
# ENHANCED STITCHING WITH FOOTER & HEADER HANDLING
# =============================================================
# Enhanced version of process_boundary with footer detection.

import json

def process_boundary(prev_content, next_content, page_num, model_name=claude_model):
    """Enhanced boundary processing with footer/header detection."""

    detector = FooterDetector()
    prev_footers = detector.analyze_page(prev_content, page_num)
    next_footers = detector.analyze_page(next_content, page_num + 1)
    detector.detected_footers.extend(prev_footers)
    detector.detected_footers.extend(next_footers)

    # Strip footers from previous page bottom (preserve footnotes)
    preserve_types = [FooterType.FOOTNOTE]
    prev_content = detector.strip_footers(prev_content, prev_footers, preserve_types)

    # Strip headers from next page top
    next_content = detector.strip_footers(next_content, next_footers, [])

    prev_lines = [l for l in prev_content.splitlines() if l.strip()]
    next_lines = [l for l in next_content.splitlines() if l.strip()]
    if not prev_lines or not next_lines:
        return prev_content, next_content

    prev_tail = '\n'.join(prev_lines[-2:])
    next_head = '\n'.join(next_lines[:3])

    prompt = (
        f"نهاية الصفحة {page_num}:\n{prev_tail}\n\n"
        f"بداية الصفحة {page_num+1}:\n{next_head}\n\n"
        "أجب بـ JSON فقط (لا تضف أي نص خارجه):\n"
        "{\"header\": نص الترويسة إن وجدت وإلا null,\n"
        " \"footer\": نص التذييل إن وجد وإلا null,\n"
        " \"join\": true إذا كانت الجملة منقطعة,\n"
        " \"footnote_span\": true إذا كان هناك هامش يمتد عبر الصفحتين\n"
        "}"
    )

    try:
        msg = client.messages.create(model=model_name, max_tokens=120,
            messages=[{"role": "user", "content": prompt}])
        raw = msg.content[0].text.strip()
        m = re.search(r'\{.*\}', raw, re.DOTALL)
        data = json.loads(m.group()) if m else {}
    except Exception as e:
        print(f"  Boundary {page_num}->{page_num+1} LLM error: {e}")
        data = {}

    # Strip detected header
    header_text = data.get('header')
    if header_text:
        lines = next_content.splitlines()
        for idx, line in enumerate(lines):
            if line.strip() and (header_text.strip() in line or line.strip() in header_text):
                lines[idx] = ''
                break
        next_content = '\n'.join(lines)

    # Strip detected footer
    footer_text = data.get('footer')
    if footer_text:
        lines = prev_content.splitlines()
        for idx in range(len(lines) - 1, -1, -1):
            line = lines[idx]
            if line.strip() and (footer_text.strip() in line or line.strip() in footer_text):
                lines[idx] = ''
                break
        prev_content = '\n'.join(lines)

    # Join split sentence
    if data.get('join') and not data.get('footnote_span'):
        next_stripped = next_content.lstrip().splitlines()
        first_idx = next((i for i, l in enumerate(next_stripped) if l.strip()), None)
        if first_idx is not None:
            continuation = next_stripped[first_idx].lstrip()
            next_stripped[first_idx] = ''
            prev_content = prev_content.rstrip() + ' ' + continuation
            next_content = '\n'.join(next_stripped)

    return prev_content, next_content


# --- Main stitching loop with footer support ---
markers, contents = [], []
for block in corrected_pages:
    lines = block.splitlines()
    if lines and re.match(r'--- Page \d+ ---', lines[0]):
        markers.append(lines[0])
        contents.append('\n'.join(lines[1:]))
    else:
        markers.append('')
        contents.append(block)

all_footers = []
all_footnotes = []
detector = FooterDetector()

print(f'\n=== Phase 1: Footer Detection ({len(contents)} pages) ===')
for i, content in enumerate(contents):
    page_num = i + 1
    footers = detector.analyze_page(content, page_num)
    all_footers.extend(footers)
    page_footnotes = [f for f in footers if f.footer_type == FooterType.FOOTNOTE]
    if page_footnotes:
        all_footnotes.append({'page': page_num, 'footnotes': [f.text for f in page_footnotes]})
    print(f"  Page {page_num}: {len(footers)} footer elements detected")
    for f in footers:
        print(f"    - {f.footer_type.value}: '{f.text[:50]}...' (conf: {f.confidence:.2f})")

print('\n=== Phase 2: Footer Stripping ===')
stripped_contents = []
for i, content in enumerate(contents):
    page_num = i + 1
    page_footers = [f for f in all_footers if f.page_num == page_num]
    stripped = detector.strip_footers(content, page_footers, [FooterType.FOOTNOTE])
    stripped_contents.append(stripped)
    removed = len([f for f in page_footers if f.footer_type != FooterType.FOOTNOTE])
    print(f"  Page {page_num}: {removed} stripped, {len(page_footers) - removed} preserved")

print(f'\n=== Phase 3: Boundary Stitching ({len(stripped_contents)-1} boundaries) ===')
for i in range(len(stripped_contents) - 1):
    print(f'  Page {i+1} -> {i+2}...')
    stripped_contents[i], stripped_contents[i+1] = process_boundary(
        stripped_contents[i], stripped_contents[i+1], i+1)

stitched_pages = [f"{markers[i]}\n{stripped_contents[i]}".strip() for i in range(len(stripped_contents))]
flowing_text = re.sub(r'\n{3,}', '\n\n',
    '\n\n'.join(c.strip() for c in stripped_contents if c.strip())).strip()

print('\nStitching complete.')
print('\n── Boundary preview (page 1 -> 2) ──')
print(stripped_contents[0].strip()[-200:])
print('  ···  ')
print(stripped_contents[1].strip()[:200])



=== Phase 1: Footer Detection (5 pages) ===
  Page 1: 1 footer elements detected
    - running_header: 'نجدة فتحى صفوة...' (conf: 0.50)
  Page 2: 1 footer elements detected
    - running_header: 'مذكرات جعفر العسكري...' (conf: 0.80)
  Page 3: 2 footer elements detected
    - footnote: '(١) كان المرحوم طارق العسكري يجيد اللغة الإنكليزية...' (conf: 0.95)
    - running_header: '03 مقدمة...' (conf: 0.80)
  Page 4: 1 footer elements detected
    - running_header: 'مذكرات جعفر العسكري...' (conf: 0.80)
  Page 5: 1 footer elements detected
    - running_header: 'مقدمة...' (conf: 0.80)

=== Phase 2: Footer Stripping ===
  Page 1: 1 stripped, 0 preserved
  Page 2: 1 stripped, 0 preserved
  Page 3: 1 stripped, 1 preserved
  Page 4: 1 stripped, 0 preserved
  Page 5: 1 stripped, 0 preserved

=== Phase 3: Boundary Stitching (4 boundaries) ===
  Page 1 -> 2...
  Page 2 -> 3...
  Page 3 -> 4...
  Page 4 -> 5...

Stitching complete.

── Boundary preview (page 1 -> 2) ──
إذا رضي عن صيغتها أخيراً عهد به

In [10]:
# =============================================================
# Cell 08
# MAIN PIPELINE
# =============================================================

# =============================================================
# CONFIGURATION & SUMMARY
# =============================================================

CONFIG = {
    'strip_page_numbers': True,
    'strip_running_headers': True,
    'strip_separators': True,
    'preserve_footnotes': True,
    'extract_footnotes_separately': True,
    'generate_footer_report': True,
}

print(f"\n=== Processing Complete ===")
print(f"Total footer elements detected: {len(all_footers)}")
print(f"Total footnotes extracted: {len(all_footnotes)}")



=== Processing Complete ===
Total footer elements detected: 6
Total footnotes extracted: 1


In [11]:
# =====================================================================
# === CELL_9_SAVE_OUTPUTS ===
# REPLACE the original save cell with this code
# =====================================================================

# --- Save all output formats ---

# 1. Page-marked corrected text
with open(os.path.join(output_dir, 'LLM_corrected_text.txt'), 'w', encoding='utf-8') as f:
    f.write('\n\n'.join(stitched_pages))
print('Saved: LLM_corrected_text.txt')

# 2. Clean flowing text
with open(os.path.join(output_dir, 'flowing_text.txt'), 'w', encoding='utf-8') as f:
    f.write(flowing_text)
print('Saved: flowing_text.txt')

# 3. Extracted footnotes (if any)
if all_footnotes and CONFIG.get('extract_footnotes_separately', True):
    footnote_text = []
    for fn_group in all_footnotes:
        footnote_text.append(f"--- Page {fn_group['page']} ---")
        for fn in fn_group['footnotes']:
            footnote_text.append(fn)
        footnote_text.append('')
    with open(os.path.join(output_dir, 'extracted_footnotes.txt'), 'w', encoding='utf-8') as f:
        f.write('\n'.join(footnote_text))
    print('Saved: extracted_footnotes.txt')

# 4. Footer detection report
if CONFIG.get('generate_footer_report', True):
    report = detector.get_footer_report()
    with open(os.path.join(output_dir, 'footer_report.txt'), 'w', encoding='utf-8') as f:
        f.write(report)
    print('Saved: footer_report.txt')

# 5. Processing summary
summary = f"""
=================================================
OCR PROCESSING SUMMARY
=================================================
Input: {pdf_file}
Pages processed: {len(contents)}

Footer Detection Results:
  - Total elements: {len(all_footers)}
  - Page numbers: {len([f for f in all_footers if f.footer_type == FooterType.PAGE_NUMBER])}
  - Footnotes: {len([f for f in all_footers if f.footer_type == FooterType.FOOTNOTE])}
  - Running headers: {len([f for f in all_footers if f.footer_type == FooterType.RUNNING_HEADER])}
  - Separators: {len([f for f in all_footers if f.footer_type == FooterType.SEPARATOR])}

Configuration:
  - Strip page numbers: {CONFIG['strip_page_numbers']}
  - Strip running headers: {CONFIG['strip_running_headers']}
  - Preserve footnotes: {CONFIG['preserve_footnotes']}

Output Files:
  - LLM_corrected_text.txt (page-marked, footers stripped)
  - flowing_text.txt (clean continuous text)
  - extracted_footnotes.txt (footnotes only)
  - footer_report.txt (detection details)
=================================================
"""
with open(os.path.join(output_dir, 'processing_summary.txt'), 'w', encoding='utf-8') as f:
    f.write(summary)
print('Saved: processing_summary.txt')
print('\n' + summary)


Saved: LLM_corrected_text.txt
Saved: flowing_text.txt
Saved: extracted_footnotes.txt
Saved: footer_report.txt
Saved: processing_summary.txt


OCR PROCESSING SUMMARY
Input: pages_5_9.pdf
Pages processed: 5

Footer Detection Results:
  - Total elements: 6
  - Page numbers: 0
  - Footnotes: 1
  - Running headers: 5
  - Separators: 0

Configuration:
  - Strip page numbers: True
  - Strip running headers: True
  - Preserve footnotes: True

Output Files:
  - LLM_corrected_text.txt (page-marked, footers stripped)
  - flowing_text.txt (clean continuous text)
  - extracted_footnotes.txt (footnotes only)
  - footer_report.txt (detection details)



In [14]:
!zip -r /content/ara-ocr.zip /content/ara-ocr

  adding: content/ara-ocr/ (stored 0%)
  adding: content/ara-ocr/footer_report.txt (stored 0%)
  adding: content/ara-ocr/processing_summary.txt (deflated 53%)
  adding: content/ara-ocr/flowing_text.txt (deflated 69%)
  adding: content/ara-ocr/optimized_approach.txt (deflated 66%)
  adding: content/ara-ocr/LLM_corrected_text.txt (deflated 69%)
  adding: content/ara-ocr/extracted_footnotes.txt (deflated 47%)
